<a href="https://colab.research.google.com/github/sanjayssnit/langchain_basics/blob/main/langchain_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install required libraries

In [ ]:
from openai import OpenAI

# Initialize the client using Anthropic's OpenAI-compatible endpoint
client = OpenAI(
    api_key="",  # Or omit to read from ANTHROPIC_API_KEY env var
    base_url="https://api.anthropic.com/v1/"
)

system_prompt = "You are a helpful coding assistant."
user_prompt = "Explain openshift platform"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model="claude-sonnet-5-5",  # Other options: claude-opus-5-5, claude-haiku-5-5, claude-fable-5-2
    messages=messages
)

print(response.choices[0].message.content)

In [ ]:
from typing import Literal
from pydantic import BaseModel
from anthropic import Anthropic

# 1. Define your Pydantic schema
class ClaimDecision(BaseModel):
    rationale: str
    decision: Literal["approve", "decline", "refer"]
    payout: float

# 2. Initialize the Anthropic client (automatically reads ANTHROPIC_API_KEY)
#client = Anthropic()
#client = OpenAI(
#    api_key=""
#)

client = Anthropic(
    api_key=""
)

# 3. Call messages.parse() with output_format
response = client.messages.parse(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Evaluate this claim: Policyholder was rear-ended at a red light. Repair estimate is $1,250. Police report confirms other driver is at fault.",
        }
    ],
    output_format=ClaimDecision,
)

# 4. Access the validated Pydantic object directly
result: ClaimDecision = response.parsed_output

print(f"Decision: {result.decision}")
print(f"Payout: ${result.payout}")
print(f"Rationale: {result.rationale}")

Decision: approve
Payout: $1250.0
Rationale: This is a straightforward liability claim with clear fault determination. The policyholder was rear-ended at a red light, which establishes the other driver's negligence. The police report confirms fault lies with the other party. The repair estimate of $1,250 is reasonable for typical rear-end collision damage. Since fault is clearly established and the claim amount is within normal parameters for this type of accident, this claim should be approved for the full repair estimate.


In [ ]:
#Using same code with LangChain

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize the native Anthropic chat model
# Omit api_key if ANTHROPIC_API_KEY is set in your environment variables
llm = ChatAnthropic(
    model="claude-sonnet-5-5",
    api_key="",
)

messages = [
    SystemMessage(content="You are a helpful coding assistant."),
    HumanMessage(content="Explain Python decorators in two sentences."),
]

response = llm.invoke(messages)
print(response.content)

A Python decorator is a function that takes another function (or class) as input and returns a modified or enhanced version of it, letting you add behavior like logging, caching, or access control without changing the original code. You apply one by placing `@decorator_name` above a function definition, which is shorthand for `func = decorator_name(func)`.


In [ ]:
import os
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

# 1. Set your API key (recommended via environment variable)

# 2. Define your desired output structure using Pydantic
class ConceptExplanation(BaseModel):
    explanation: str = Field(
        description="A clear, two-sentence explanation of the requested concept."
    )
    example_snippet: str = Field(
        description="A short, runnable Python code example demonstrating the concept."
    )

# 3. Initialize the Anthropic model
# Note: Use valid Anthropic model strings (e.g., claude-sonnet-4-5-20250929 or claude-3-5-sonnet-latest)
llm = ChatAnthropic(
    model="claude-sonnet-5-5",
    api_key="",
)

# 4. Bind the Pydantic schema to the model
structured_llm = llm.with_structured_output(ConceptExplanation)

# 5. Format your messages using LangChain message objects
messages = [
    SystemMessage(content="You are a helpful coding assistant."),
    HumanMessage(content="Explain Python decorators in two sentences.")
]

# 6. Invoke the model (returns an instance of ConceptExplanation)
response = structured_llm.invoke(messages)

# Access the structured fields directly as Python attributes
print(f"Explanation:\n{response.explanation}\n")
print(f"Code Example:\n{response.example_snippet}")

C:\Users\e45608\AppData\Roaming\Python\Python312\site-packages\langchain_anthropic\chat_models.py:2992: UserWarning: You are attempting to use structured output via forced tool calling, which is not supported when `thinking` is enabled or on claude-sonnet-5-5. This method will raise an OutputParserException if tool calls are not generated. Consider `method='json_schema'`, disabling `thinking` where supported, or adjusting your prompt to ensure the tool is called.
  llm = self._get_llm_for_structured_output_when_thinking_is_enabled(


Explanation:
A Python decorator is a function that takes another function as input and returns a new function that extends or modifies its behavior without changing its source code. It is applied using the @decorator_name syntax placed above the function definition, which is shorthand for reassigning the function to the decorator's result.

Code Example:
import functools

def shout(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    return f"hello, {name}!"

print(greet("world"))  # HELLO, WORLD!


In [1]:
!pip install langchain langchain-openai langchain-community

Defaulting to user installation because normal site-packages is not writeable
  Using cached openai-2.54.0-py3-none-any.whl.metadata (40 kB)
Using cached openai-2.54.0-py3-none-any.whl (1.7 MB)
  Attempting uninstall: openai
    Found existing installation: openai 3.46.0
    Uninstalling openai-3.46.0:
      Successfully uninstalled openai-3.46.0



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configure the LiteLLM Gateway
Set LITELLM_API_BASE to the gateway's OpenAI-compatible base URL ending in /v1, LITELLM_API_KEY to its key, and LITELLM_MODEL to the gateway model alias routed to Claude before starting the notebook.

In [8]:
import os

from dotenv import load_dotenv, find_dotenv

env_file = find_dotenv(usecwd=True)

print("Found:", env_file)

load_dotenv(env_file)

required_settings = ("LITELLM_API_BASE", "LITELLM_API_KEY", "LITELLM_MODEL")
missing_settings = [name for name in required_settings if not os.environ.get(name)]
if missing_settings:
    raise RuntimeError(
        "Set these environment variables before running this notebook: "
        + ", ".join(missing_settings)
    )

LITELLM_API_BASE = os.environ["LITELLM_API_BASE"].rstrip("/")
LITELLM_API_KEY = os.environ["LITELLM_API_KEY"]
LLM_MODEL = os.environ["LITELLM_MODEL"]

def _openai_base_url() -> str:
    return LITELLM_API_BASE

Found: C:\AI\lanchain-examples\.env


### Chat without memory
A simple chat with LLM without using memory to store conversation history. The LLM will treat each question separately as it wont understand the previous conversations and the complete context.

In [41]:
from langchain_openai import ChatOpenAI

def get_llm(temperature: float = 1.0, max_tokens: int = 1024) -> ChatOpenAI:
    """Chat client that calls the gateway's /v1/chat/completions route."""
    return ChatOpenAI(
        model=LLM_MODEL,
        base_url=_openai_base_url(),
        api_key=LITELLM_API_KEY,
        temperature=temperature,
        max_tokens=max_tokens,
        # Ask the gateway to include token usage so the semantic cache can
        # track cost savings.
        stream_usage=True,
    )

llm = get_llm()

def simple_chat_without_memory(user_query):
  response = llm.invoke(user_query)
  return response.content

In [42]:
response1 = simple_chat_without_memory("I want to learn AI")
print(response1)

# Getting Started with AI

Great choice! Here's a practical roadmap to help you begin.

## First, Clarify Your Goal
AI is broad. Which direction interests you?
- **Building AI applications** (using existing models/APIs)
- **Machine learning engineering** (training models)
- **Research** (developing new methods)
- **Just understanding** how it works conceptually

Your answer changes the path significantly.

## Foundational Skills

**Math (for deeper work):**
- Linear algebra, calculus, probability & statistics

**Programming:**
- **Python** is the standard language
- Key libraries: NumPy, Pandas, then PyTorch or TensorFlow

## A Suggested Learning Path

1. **Python basics** (if needed)
2. **Core ML concepts** — supervised/unsupervised learning, overfitting, etc.
3. **Hands-on projects** — learn by building
4. **Specialize** — deep learning, NLP, computer vision, etc.

## Recommended Free Resources

| Resource | Best For |
|----------|----------|
| Andrew Ng's *Machine Learning* (Courser

In [43]:
response2 = simple_chat_without_memory("Can you give me a learning plan?")
print(response2)

I'd be happy to help you create a learning plan! But first, I need a bit more information to make it useful for you.

Could you tell me:

1. **What do you want to learn?** (a language, a skill like coding or drawing, a subject like math or history, etc.)

2. **What's your current level?** (complete beginner, some experience, intermediate, etc.)

3. **How much time can you commit?** (hours per week, and over what overall timeframe)

4. **What's your goal?** (career change, hobby, passing an exam, a specific project, general curiosity)

5. **How do you learn best?** (videos, reading, hands-on projects, courses, etc.)

Once I know these, I can put together a focused, realistic plan for you.

If you'd rather just get started, here's a **general learning plan template** you can adapt to anything:

## General Learning Plan Template

**Phase 1: Foundations (Weeks 1–2)**
- Understand the basics and key terminology
- Find 1–2 quality resources (don't overload)
- Set a clear, measurable goal

**

### Chat with memory managed using internal List
A simple chat with LLM with basic memory management to store conversation history. The LLM will read each question along with the previous conversation and answer based on the complete context.

In [44]:
from langchain_core.messages import HumanMessage, AIMessage

conversation = []

def simple_chat_with_memory(user_query):
  conversation.append(HumanMessage(content=user_query))
  response = llm.invoke(conversation)
  conversation.append(AIMessage(content=response.content))
  return response.content

In [45]:
response1 = simple_chat_with_memory("I want to learn AI")
print(response1)

# Learning AI: A Roadmap

Great goal! Here's a practical path to get started.

## First, Clarify Your Goal
AI is broad. What interests you most?
- **Building AI applications** (using existing tools/APIs)
- **Machine learning engineering** (training models)
- **Research** (developing new algorithms)
- **Just understanding** how it works

Your answer changes the best path significantly.

## Foundational Skills

**1. Programming** – Python is the standard
- Learn basics: variables, loops, functions, data structures

**2. Math** (depth depends on your goal)
- Linear algebra, statistics, calculus
- You can learn these alongside, not before, coding

## Suggested Learning Path

| Stage | Focus | Resources |
|-------|-------|-----------|
| 1 | Python basics | freeCodeCamp, Python docs |
| 2 | ML concepts | Andrew Ng's ML course (Coursera) |
| 3 | Hands-on | Kaggle, scikit-learn |
| 4 | Deep learning | fast.ai, DeepLearning.AI |
| 5 | Specialize | NLP, computer vision, LLMs, etc. |

## Beginner

In [ ]:
print(conversation)

In [46]:
response2 = simple_chat_with_memory("Can you give me a learning plan?")
print(response2)

# Your AI Learning Plan

Here's a structured 6-month plan. Adjust the pace to your schedule (this assumes ~10 hrs/week).

---

## Month 1: Python Foundations
**Goal:** Get comfortable writing code

- [ ] Python basics: variables, loops, functions, lists, dictionaries
- [ ] Libraries: NumPy, Pandas (data handling)
- [ ] Practice: small scripts, simple data analysis

**Resources:** freeCodeCamp Python course, Kaggle "Python" + "Pandas" micro-courses

---

## Month 2: Math + Data Essentials
**Goal:** Understand the concepts behind ML (lightweight, applied)

- [ ] Statistics basics: mean, variance, distributions, probability
- [ ] Linear algebra basics: vectors, matrices
- [ ] Data visualization: Matplotlib, Seaborn

**Resources:** Khan Academy (stats & linear algebra), StatQuest (YouTube)

> 💡 Don't over-invest here. Learn enough to move forward, revisit as needed.

---

## Month 3: Core Machine Learning
**Goal:** Understand how ML actually works

- [ ] Key concepts: training/testing, ove

### Chat prompt template and Chaining
Langchain's chaining concept using Chat prompt template and an LLM invocation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate(messages=[
    ("system", "You are a helpful personal assistant"),
    ("human", "{user_query}")
])

chain = prompt | llm
response_chain = chain.invoke({"user_query":"I have decided to learn AI"})
print(response_chain)

In [ ]:
response_chain = chain.invoke({"user_query":"Can you give me a learning plan?"})
print(response_chain)

### Cahining with memory managed in prompt template

In [19]:
prompt_with_memory = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful personal assistant"),
    ("placeholder", "{chat_history}"),
    ("human", "{user_query}")
])

chain_with_memory = prompt_with_memory | llm

In [20]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}
# Function to get user's conversation history based on the user's session id
def get_history(session_id: str):
  if session_id not in store:
    store[session_id] = ChatMessageHistory()
  return store[session_id]

C:\Users\e45608\AppData\Local\Temp\ipykernel_58688\3815019727.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [47]:
from langchain_core.runnables import RunnableWithMessageHistory

# Runnable with message history enables us to run a specified chain and
# gets the user's history using a callback function.

chat_with_memory = RunnableWithMessageHistory(
    runnable=chain_with_memory,
    get_session_history=get_history,
    input_messages_key="user_query",
    history_messages_key="chat_history"
)

response = chat_with_memory.invoke({"user_query":"I want to learn AI"}, {"configurable": {"session_id": "user1"}})
print(response)

C:\Users\e45608\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


content="It looks like we might be going in a bit of a loop! 😊 I want to make sure I actually help you move forward.\n\nI already shared a **6-month learning plan** in my last message — did that not quite hit the mark? Let me know what you're looking for:\n\n- **Was the plan too long/detailed?** I can give you a simpler, shorter version.\n- **Too generic?** Tell me a bit about yourself and I'll tailor it.\n- **Not sure where to start?** I can just give you **one single first step** to take today.\n\n---\n\n### Let's try the simplest possible path. Just pick a number:\n\n**1.** I want to **use AI tools** (like ChatGPT) better — no coding\n**2.** I want to **build AI apps** — some coding involved\n**3.** I want to **deeply understand AI** — career/ML engineer path\n**4.** I'm **not sure** — just help me get started\n\nReply with **1, 2, 3, or 4**, and I'll give you exactly what to do first. 👍" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 320

In [48]:
response2 = chat_with_memory.invoke({"user_query":"Can you give me a learning plan"}, {"configurable": {"session_id": "user1"}})
print(response2)

content='I notice we keep circling back to the same request — so let me just **give you the simplest possible plan right now**, no questions asked. You can adjust later.\n\n---\n\n## 🚀 Your Simple AI Learning Plan\n\n### **Step 1: This Week**\n- Create a free account on **ChatGPT** or **Claude**\n- Spend 30 min/day just *using* it — ask questions, give it tasks\n- 📺 Watch: *"Elements of AI"* free course (elementsofai.com)\n\n### **Step 2: Next 2 Weeks**\n- Learn **Python basics** (the language of AI)\n- 📺 Free resource: **freeCodeCamp Python course** on YouTube (4 hours)\n- Just follow along and type the code yourself\n\n### **Step 3: Weeks 4–8**\n- Take **Andrew Ng\'s "AI for Everyone"** (Coursera, free to audit)\n- Then start **"Machine Learning Specialization"** by Andrew Ng\n- Learn basic tools: **NumPy, Pandas**\n\n### **Step 4: Months 3–4**\n- Do the **fast.ai** course (practical deep learning)\n- Build your **first project** (e.g., image classifier or simple predictor)\n\n### **

In [49]:
# Question from a different user is get's conversation history only for that user.
response3 = chat_with_memory.invoke({"user_query":"Can you give me a learning plan"}, {"configurable": {"session_id": "user2"}})
print(response3)

content="I'd be happy to help you create a learning plan! But first, I need a bit more information to make it useful for you.\n\nCould you tell me:\n\n1. **What do you want to learn?** (e.g., a programming language, a musical instrument, a new language, cooking, data science, etc.)\n\n2. **What's your current level?** (complete beginner, some experience, intermediate, etc.)\n\n3. **What's your goal?** (hobby, career change, pass an exam, build a specific project, etc.)\n\n4. **How much time can you commit?** (e.g., 1 hour/day, a few hours on weekends, etc.)\n\n5. **What's your timeframe?** (e.g., 1 month, 6 months, no deadline, etc.)\n\n6. **How do you learn best?** (videos, reading, hands-on projects, courses, etc.)\n\nOnce you share these details, I can put together a personalized, step-by-step plan for you. \n\nIf you'd like, you can also just tell me the topic and I'll draft a general plan you can adjust!" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'comp